# Renewable Energy Siting — Exploratory Analysis

This notebook walks through exploratory data analysis of the candidate site dataset prior to running the full ML + MCDA pipeline.

**Contents:**
1. Load and inspect data
2. Distribution of spatial attributes
3. Correlation analysis
4. Geographic distribution of sites
5. Class balance check
6. Quick Random Forest baseline

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
gdf = gpd.read_file('../data/processed/all_candidate_sites.gpkg')
print(f'Shape: {gdf.shape}')
print(f'CRS:   {gdf.crs}')
gdf.head(3)

In [ ]:
gdf.describe().T.round(3)

## 2. Attribute Distributions

In [ ]:
numeric_cols = ['resource_value', 'slope_pct', 'dist_road_km',
                'dist_transmission_km', 'elevation_m',
                'population_density', 'grid_capacity_mw']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for tech, color in [('wind', '#4fc3f7'), ('solar', '#ffb74d')]:
        sub = gdf[gdf['tech_type'] == tech][col].dropna()
        axes[i].hist(sub, bins=50, alpha=0.6, color=color, label=tech, density=True)
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Attribute Distributions — Wind vs Solar', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Class Balance

In [ ]:
balance = gdf.groupby(['tech_type', 'suitable']).size().unstack(fill_value=0)
balance.columns = ['Not Suitable', 'Suitable']
balance['Pct Suitable'] = (balance['Suitable'] / balance.sum(axis=1) * 100).round(1)
print(balance)

fig, ax = plt.subplots(figsize=(8, 4))
balance[['Not Suitable', 'Suitable']].plot(kind='bar', ax=ax,
    color=['#e74c3c', '#00d4aa'], edgecolor='none', alpha=0.85)
ax.set_title('Class Balance by Technology')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap (Wind)

In [ ]:
wind = gdf[gdf['tech_type'] == 'wind'][numeric_cols + ['suitable']].dropna()

fig, ax = plt.subplots(figsize=(10, 8))
corr = wind.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix — Wind Sites', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Quick RF Baseline

In [ ]:
feat_cols = ['resource_value', 'slope_pct', 'dist_road_km',
             'dist_transmission_km', 'protected_area', 'land_use_code',
             'elevation_m', 'population_density', 'setback_met', 'grid_capacity_mw']

wind_clean = gdf[gdf['tech_type'] == 'wind'].dropna(subset=feat_cols)
X = wind_clean[feat_cols]
y = wind_clean['suitable']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                             n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}')
print(classification_report(y_test, y_pred,
      target_names=['Not Suitable', 'Suitable']))

## 6. Geographic Preview (Sample)

In [ ]:
# Sample for quick plot
sample = gdf.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_facecolor('#0d1b2a')
sample[sample['tech_type'] == 'wind'].plot(
    ax=ax, column='resource_value', cmap='Blues', alpha=0.6, markersize=1,
    legend=False)
sample[sample['tech_type'] == 'solar'].plot(
    ax=ax, column='resource_value', cmap='Oranges', alpha=0.6, markersize=1,
    legend=False)
ax.set_title('Sample of Candidate Sites — Resource Value (Blue=Wind, Orange=Solar)',
             fontsize=12)
plt.tight_layout()
plt.show()